# 11-2. 종일 음이항 NB2 모형 튜닝

2025년 종일 서울행 광역버스 승차인원을 종속변수로 하고 `log(2024년 승차인원)`을 offset으로 적용한다. 분석 단위는 셀×노선이며 노선 고정효과와 셀 단위 군집 강건 표준오차를 사용한다. 모든 연속형 변수는 표준화하지 않은 원단위이다.

- 기본모형: 보행 더미 2개 + IC + 경쟁 목적지 비율 + 2024년 20~50대 인구수 + 건축물 수
- 튜닝 1: 보행 20분 이내 더미만 사용하고 나머지는 기본모형과 동일
- 튜닝 2: 기본모형의 2024년 인구수를 2024→2025년 20~50대 인구 증감으로 교체


In [33]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
import config as cfg
cfg = importlib.reload(cfg)
cfg.setup_notebook()
ID, ROUTE = '셀 ID', '노선'
print('설정 파일:', Path(cfg.__file__).resolve())
print('경쟁목적지 파일:', cfg.COMPETING_CELL_ROUTE_PATH)


설정 파일: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\config.py
경쟁목적지 파일: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\data\processed\05_GTX_경쟁목적지비율_종일_셀노선.csv


## 1. 공통 데이터 구축

In [34]:
grid = cfg.load_model_data()
routes = cfg.load_cell_route_all_day()
grid_cols = [ID, 'walk_20', 'walk_20_60', 'ic_access', 'pop_20_50', 'buildings']
model_data = routes.merge(grid[grid_cols], on=ID, how='inner', validate='many_to_one')

pop = cfg.read_csv(cfg.POPULATION_CHANGE_PATH)
pop[ID] = pd.to_numeric(pop[ID], errors='coerce').astype('Int64')
pop_col = next(c for c in pop.columns if '증감(100명)' in str(c))
pop['pop_change'] = pd.to_numeric(pop[pop_col], errors='coerce') * 100
model_data = model_data.merge(pop[[ID, 'pop_change']], on=ID, how='left', validate='many_to_one')

numeric_cols = ['y2024','y2025','walk_20','walk_20_60','ic_access','competing_share','pop_20_50','pop_change','buildings']
for c in numeric_cols:
    model_data[c] = pd.to_numeric(model_data[c], errors='coerce')
model_data = model_data[model_data['y2024'] > 0].dropna(subset=numeric_cols).reset_index(drop=True)
print('관측치:',len(model_data),'| 격자:',model_data[ID].nunique(),'| 노선:',model_data[ROUTE].nunique())
print('셀×노선 중복:',model_data.duplicated([ID,ROUTE]).sum())
print('경쟁 목적지 비율 결측:',model_data['competing_share'].isna().sum())
display(model_data[[ID,ROUTE]+numeric_cols].head())


관측치: 220 | 격자: 132 | 노선: 24
셀×노선 중복: 0
경쟁 목적지 비율 결측: 0


,셀 ID,노선,y2024,y2025,walk_20,walk_20_60,ic_access,competing_share,pop_20_50,pop_change,buildings
0,44,1200,269,164,0.0,1.0,5.071,21.189591,937.0,53.0,18.0
1,76,2200,17,35,0.0,0.0,1.921,0.000000,33.0,7.0,37.0
2,559,3100,82,90,0.0,1.0,0.319,0.000000,0.0,0.0,46.0
3,619,9710,1,4,0.0,0.0,9.029,100.000000,0.0,0.0,7.0
4,765,9709,12,14,0.0,0.0,4.323,25.000000,139.0,0.0,92.0


## 2. NB2 추정 함수

In [35]:
LABELS={'walk_20':'GTX 보행 20분 이내','walk_20_60':'GTX 보행 20~60분','ic_access':'IC까지 도로거리(km)','competing_share':'GTX 경쟁 목적지 비율(%p)','pop_20_50':'2024년 20~50대 인구(명)','pop_change':'2024→2025년 20~50대 인구 증감(명)','buildings':'2024년 건축물 수(동)'}

def fit_nb2(name, features):
    d=model_data.dropna(subset=['y2024','y2025']+features).copy()
    x=d[features].astype(float)  # 원단위, 표준화 없음
    route_fe=pd.get_dummies(d[ROUTE].astype(str),prefix='노선',drop_first=True,dtype=float)
    X=sm.add_constant(pd.concat([x,route_fe],axis=1),has_constant='add')
    offset=np.log(d['y2024'].astype(float))
    poisson=sm.GLM(d['y2025'],X,family=sm.families.Poisson(),offset=offset).fit()
    model=NegativeBinomial(d['y2025'],X,offset=offset,loglike_method='nb2')
    fit=model.fit(start_params=np.r_[np.asarray(poisson.params),0.2],method='bfgs',maxiter=5000,disp=False,cov_type='cluster',cov_kwds={'groups':pd.factorize(d[ID])[0],'use_correction':True})
    if not fit.mle_retvals.get('converged') or fit.mle_retvals.get('warnflag',0)!=0:
        raise RuntimeError(f'{name} 수렴 실패: {fit.mle_retvals}')
    null=NegativeBinomial(d['y2025'],np.ones((len(d),1)),offset=offset,loglike_method='nb2').fit(start_params=np.array([np.log(d['y2025'].sum()/d['y2024'].sum()),0.2]),method='bfgs',maxiter=5000,disp=False)
    ci=fit.conf_int().loc[features]
    result=pd.DataFrame({'변수':[LABELS[v] for v in features],'계수':[fit.params[v] for v in features],'IRR':[np.exp(fit.params[v]) for v in features],'상대적 차이(%)':[(np.exp(fit.params[v])-1)*100 for v in features],'z값':[fit.tvalues[v] for v in features],'p-value':[fit.pvalues[v] for v in features],'95% CI 하한(%)':[(np.exp(ci.loc[v,0])-1)*100 for v in features],'95% CI 상한(%)':[(np.exp(ci.loc[v,1])-1)*100 for v in features]})
    info={'모형':name,'표본 수':len(d),'격자 수':d[ID].nunique(),'노선 수':d[ROUTE].nunique(),'로그우도':fit.llf,'McFadden pseudo R²':1-fit.llf/null.llf,'AIC':fit.aic,'BIC':fit.bic,'alpha':fit.params['alpha'],'수렴':fit.mle_retvals.get('converged')}
    return {'fit':fit,'data':d,'X':X,'offset':offset,'result':result,'info':info}


## 3. 기본모형 — 보행 더미 2개와 2024년 인구수

In [36]:
BASE_FEATURES=['walk_20','walk_20_60','ic_access','competing_share','pop_20_50','buildings']
base_model=fit_nb2('기본모형',BASE_FEATURES)
display(pd.DataFrame([base_model['info']]).round(4))
display(base_model['result'].round(4))


,모형,표본 수,격자 수,노선 수,로그우도,McFadden pseudo R²,AIC,BIC,alpha,수렴
0,기본모형,220,132,24,-937.1742,0.053,1936.3483,2041.5508,0.2235,True


,변수,계수,IRR,상대적 차이(%),z값,p-value,95% CI 하한(%),95% CI 상한(%)
0,GTX 보행 20분 이내,-0.8642,0.4214,-57.8592,-3.6441,0.0003,-73.5241,-32.9260
1,GTX 보행 20~60분,-0.5148,0.5976,-40.2409,-2.5922,0.0095,-59.5102,-11.8012
2,IC까지 도로거리(km),-0.0763,0.9266,-7.3425,-2.7277,0.0064,-12.2833,-2.1235
3,GTX 경쟁 목적지 비율(%p),0.0068,1.0068,0.6827,1.3113,0.1898,-0.3360,1.7118
4,2024년 20~50대 인구(명),-0.0001,0.9999,-0.0079,-0.6701,0.5028,-0.0309,0.0151
5,2024년 건축물 수(동),-0.0031,0.9969,-0.3117,-1.6998,0.0892,-0.6699,0.0478


### 3-1. 기본모형 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

font_candidates=[Path(r'C:\\Windows\\Fonts\\malgun.ttf'),Path(r'C:\\Windows\\Fonts\\malgunbd.ttf')]
font_path=next((p for p in font_candidates if p.exists()),None)
font_prop=None
if font_path is not None:
    fm.fontManager.addfont(str(font_path))
    font_prop=fm.FontProperties(fname=str(font_path))
    plt.rcParams['font.family']=font_prop.get_name()
plt.rcParams['axes.unicode_minus']=False

fit=base_model['fit']
features=BASE_FEATURES
ci=fit.conf_int().loc[features]
plot_table=pd.DataFrame({
    '변수':[LABELS[v] for v in features],
    'IRR':[np.exp(fit.params[v]) for v in features],
    'IRR 하한':[np.exp(ci.loc[v,0]) for v in features],
    'IRR 상한':[np.exp(ci.loc[v,1]) for v in features],
    '상대적 승차 차이(%)':[(np.exp(fit.params[v])-1)*100 for v in features],
    'p-value':[fit.pvalues[v] for v in features]
}).iloc[::-1].reset_index(drop=True)

y=np.arange(len(plot_table))
irr=plot_table['IRR'].to_numpy()
lower=plot_table['IRR 하한'].to_numpy()
upper=plot_table['IRR 상한'].to_numpy()
xerr=np.vstack([irr-lower,upper-irr])
colors=['#d73027' if p<0.05 else '#f4a742' if p<0.10 else '#9e9e9e' for p in plot_table['p-value']]

fig,ax=plt.subplots(figsize=(13.5,6.2))
ax.barh(y,irr-1,left=1,color=colors,alpha=0.82,height=0.55)
ax.errorbar(irr,y,xerr=xerr,fmt='o',color='black',ecolor='black',capsize=4,elinewidth=1.4,markersize=4)
ax.axvline(1,color='black',linestyle='--',linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(plot_table['변수'].tolist(),fontproperties=font_prop,fontsize=11)
ax.set_xlabel('IRR',fontproperties=font_prop,fontsize=11)
x_min=max(0,float(lower.min())-0.08)
x_max=max(1.05,float(upper.max())+0.03)
ax.set_xlim(x_min,x_max)
for i,row in plot_table.iterrows():
    effect=row['상대적 승차 차이(%)']
    sign='+' if effect>=0 else ''
    mark='***' if row['p-value']<0.01 else '**' if row['p-value']<0.05 else '*' if row['p-value']<0.10 else ''
    ax.text(1.04,i,f"IRR {row['IRR']:.4f}  |  {sign}{effect:.2f}% {mark}",transform=ax.get_yaxis_transform(),ha='left',va='center',fontproperties=font_prop,fontsize=10.5,clip_on=False)
ax.grid(axis='x',alpha=0.22)
ax.grid(axis='y',visible=False)
ax.tick_params(axis='x',labelsize=10)
plt.subplots_adjust(left=0.25,right=0.67,bottom=0.14,top=0.98)
plt.show()
display(plot_table[['변수','IRR','상대적 승차 차이(%)','p-value','IRR 하한','IRR 상한']].round(4))


## 4. 튜닝 1 — 보행 20분 이내 더미만 사용

In [37]:
TUNE1_FEATURES=['walk_20','ic_access','competing_share','pop_20_50','buildings']
tune1_model=fit_nb2('튜닝1_20분더미만',TUNE1_FEATURES)
display(pd.DataFrame([tune1_model['info']]).round(4))
display(tune1_model['result'].round(4))


,모형,표본 수,격자 수,노선 수,로그우도,McFadden pseudo R²,AIC,BIC,alpha,수렴
0,튜닝1_20분더미만,220,132,24,-943.5751,0.0466,1947.1501,2048.9589,0.2346,True


,변수,계수,IRR,상대적 차이(%),z값,p-value,95% CI 하한(%),95% CI 상한(%)
0,GTX 보행 20분 이내,-0.3537,0.7021,-29.7916,-2.7311,0.0063,-45.5309,-9.5042
1,IC까지 도로거리(km),-0.0412,0.9596,-4.0370,-1.4573,0.1450,-9.2107,1.4316
2,GTX 경쟁 목적지 비율(%p),0.0062,1.0063,0.6263,1.2089,0.2267,-0.3871,1.6500
3,2024년 20~50대 인구(명),-0.0001,0.9999,-0.0109,-0.9202,0.3575,-0.0341,0.0123
4,2024년 건축물 수(동),-0.0030,0.9970,-0.2968,-1.5754,0.1152,-0.6647,0.0726


## 5. 튜닝 2 — 기존 인구수를 2024→2025년 인구 증감으로 교체

In [38]:
TUNE2_FEATURES=['walk_20','walk_20_60','ic_access','competing_share','pop_change','buildings']
tune2_model=fit_nb2('튜닝2_인구증감',TUNE2_FEATURES)
display(pd.DataFrame([tune2_model['info']]).round(4))
display(tune2_model['result'].round(4))


,모형,표본 수,격자 수,노선 수,로그우도,McFadden pseudo R²,AIC,BIC,alpha,수렴
0,튜닝2_인구증감,220,132,24,-937.5268,0.0527,1937.0536,2042.256,0.2234,True


,변수,계수,IRR,상대적 차이(%),z값,p-value,95% CI 하한(%),95% CI 상한(%)
0,GTX 보행 20분 이내,-0.8871,0.4119,-58.8140,-3.8771,0.0001,-73.6976,-35.5082
1,GTX 보행 20~60분,-0.5203,0.5944,-40.5636,-2.6242,0.0087,-59.7006,-12.3389
2,IC까지 도로거리(km),-0.0763,0.9265,-7.3493,-2.7286,0.0064,-12.2926,-2.1273
3,GTX 경쟁 목적지 비율(%p),0.0067,1.0068,0.6762,1.2770,0.2016,-0.3598,1.7230
4,2024→2025년 20~50대 인구 증감(명),0.0001,1.0001,0.0119,0.2909,0.7712,-0.0682,0.0921
5,2024년 건축물 수(동),-0.0030,0.9970,-0.2955,-1.6468,0.0996,-0.6461,0.0563


## 6. 세 모형 비교

In [39]:
models=[base_model,tune1_model,tune2_model]
comparison=pd.DataFrame([m['info'] for m in models])
display(comparison.round(4))
rows=[]
for m in models:
    t=m['result'][['변수','계수','IRR','상대적 차이(%)','p-value']].copy()
    t.insert(0,'모형',m['info']['모형'])
    rows.append(t)
display(pd.concat(rows,ignore_index=True).round(4))
print('AIC·BIC는 같은 표본의 모형끼리 낮을수록 좋습니다.')
print('변수 채택은 적합도뿐 아니라 연구 가설, 계수 방향, 신뢰구간과 p-value를 함께 판단합니다.')


,모형,표본 수,격자 수,노선 수,로그우도,McFadden pseudo R²,AIC,BIC,alpha,수렴
0,기본모형,220,132,24,-937.1742,0.0530,1936.3483,2041.5508,0.2235,True
1,튜닝1_20분더미만,220,132,24,-943.5751,0.0466,1947.1501,2048.9589,0.2346,True
2,튜닝2_인구증감,220,132,24,-937.5268,0.0527,1937.0536,2042.2560,0.2234,True


,모형,변수,계수,IRR,상대적 차이(%),p-value
0,기본모형,GTX 보행 20분 이내,-0.8642,0.4214,-57.8592,0.0003
1,기본모형,GTX 보행 20~60분,-0.5148,0.5976,-40.2409,0.0095
2,기본모형,IC까지 도로거리(km),-0.0763,0.9266,-7.3425,0.0064
3,기본모형,GTX 경쟁 목적지 비율(%p),0.0068,1.0068,0.6827,0.1898
4,기본모형,2024년 20~50대 인구(명),-0.0001,0.9999,-0.0079,0.5028
5,기본모형,2024년 건축물 수(동),-0.0031,0.9969,-0.3117,0.0892
6,튜닝1_20분더미만,GTX 보행 20분 이내,-0.3537,0.7021,-29.7916,0.0063
7,튜닝1_20분더미만,IC까지 도로거리(km),-0.0412,0.9596,-4.0370,0.1450
8,튜닝1_20분더미만,GTX 경쟁 목적지 비율(%p),0.0062,1.0063,0.6263,0.2267
9,튜닝1_20분더미만,2024년 20~50대 인구(명),-0.0001,0.9999,-0.0109,0.3575


AIC·BIC는 같은 표본의 모형끼리 낮을수록 좋습니다.
변수 채택은 적합도뿐 아니라 연구 가설, 계수 방향, 신뢰구간과 p-value를 함께 판단합니다.
